# 2장 1강: 텍스트 임베딩과 의미 유사도 — 실습문제


## 실습 목표

- TF-IDF와 텍스트 임베딩의 공통점과 차이를 설명할 수 있다.
- 내적과 벡터 크기를 구하고 코사인 유사도를 계산할 수 있다.
- 질문 벡터와 문서 벡터를 비교하여 Top-K 후보를 선택할 수 있다.
- 선택된 리뷰의 원문과 근거 표현을 확인하고, 점수와 실제 의미를 구분할 수 있다.
- 유사 문서 검색, BERTopic의 토픽 생성, RAG의 역할을 구분할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas, NumPy
- `reviews.csv`, `review_labels.csv`, `aspect_labels.csv`를 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 앞선 강의의 노트북을 실행하지 않아도 됩니다. 이 파일의 준비 코드부터 위에서 아래로 실행하세요.

| 파일 | 한 행의 의미 | 이번 실습의 역할 |
|---|---|---|
| reviews.csv | 리뷰 한 개 | 후보 리뷰의 ID, 본문, 상품명·분야 등 확인 |
| review_labels.csv | 리뷰 한 개의 기존 전체 라벨 | 선택한 리뷰의 기존 코드 조회 |
| aspect_labels.csv | 리뷰의 속성별 주석 한 개 | 속성명과 evidence_text를 통해 세부 평가·조건 확인 |

세 파일은 `review_id`로 연결됩니다. `review_text`는 원문, `source`·`domain`·`category`·`product_name`은 문서를 설명하는 정보입니다. 속성별 파일의 `aspect_no`는 리뷰 안의 주석 번호이며, 같은 리뷰 ID에 여러 행이 있을 수 있습니다. 기존 `sentiment_code`는 이번 실습에서 예측한 결과도, 코사인 유사도도 아닙니다.

### 반드시 구분할 점: 실제 데이터와 가상 벡터

교안의 Python 예시는 **실제 임베딩 모델/API를 사용하지 않고 미리 정한 2차원 벡터**로 계산합니다. 이 실습도 그 범위를 따릅니다.

- 리뷰 본문·ID·상품명·기존 라벨·근거 문장은 제공된 CSV의 실제 값입니다.
- 질문 벡터와 문서 벡터는 계산 연습을 위해 사람이 정한 **가상 값**입니다. 리뷰를 읽어 모델이 생성한 값이 아닙니다.
- 벡터의 첫째·둘째 숫자에 “보온성”, “사이즈” 같은 의미를 임의로 붙이지 않습니다.
- 계산되는 점수와 순위는 **가상 벡터에 대한 수학적 결과**입니다. 실제 리뷰의 의미 유사도 성능을 측정한 결과가 아닙니다.
- 실제 리뷰를 사용하는 목적은 **벡터 행과 문서 ID 연결 → 순위 선택 → 원문·근거 확인**의 흐름을 연습하는 것입니다.

실제 임베딩 모델 다운로드·호출, BERTopic 실행, c-TF-IDF 계산, RAG 구축은 요구하지 않습니다. BERTopic과 RAG는 교안에 제시된 순서·역할·해석상의 주의점을 설명하는 수준으로 다룹니다.


## 시작 전에 읽는 개념 정리

‘배송이 빨라서 만족했다’와 ‘상품이 금방 도착해서 좋았다’는 표현이 달라도 비슷한 상황을 이야기합니다. 텍스트 임베딩은 **모델이 학습한 특징을 이용하여 글을 숫자들의 묶음으로 표현하는 방법**입니다. 비교 가능한 벡터를 만든 뒤 유사도를 구하면 관련 문서를 찾는 데 활용할 수 있습니다.

이번에는 모델을 실행하지 않습니다. 사람이 정한 작은 벡터로 **벡터 비교 → 순위 선택 → 문서 ID 연결 → 원문 확인**을 연습합니다. 임베딩이 어떤 일을 하는지 이해하는 것과 실제 임베딩 모델을 실행하는 것은 구분합니다.

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습의 예 |
|---|---|---|
| 벡터 | 순서를 가진 숫자들의 묶음 | `[0.6, 0.8]` |
| 차원 | 벡터에 들어 있는 숫자의 개수 | 숫자 두 개이므로 2차원 |
| 좌표·성분 | 벡터의 각 위치에 있는 숫자 | 첫째 성분 0.6, 둘째 성분 0.8 |
| 텍스트 임베딩 | 텍스트의 특징을 모델로 수치 표현하는 방법 | 실제 활용에서는 질문·문서를 비교 가능한 벡터로 변환 |
| 내적 | 같은 위치의 숫자끼리 곱한 뒤 더한 값 | `[a,b]`와 `[c,d]`의 내적은 `a×c+b×d` |
| 벡터 크기 | 숫자 배열이 나타내는 화살표의 길이 | 2차원에서는 `√(a²+b²)` |
| 방향 | 화살표가 향하는 쪽 | 크기가 달라도 같은 쪽을 향할 수 있음 |
| 코사인 유사도 | 두 벡터의 방향이 얼마나 비슷한지 나타내는 값 | 내적을 두 벡터 크기의 곱으로 나눔 |
| 후보 문서 | 검색 대상으로 준비한 문서 | 이번에는 실제 리뷰 3개 |
| Top-K | 후보 중 점수가 높은 K개를 선택하는 방식 | K=2이면 두 후보 선택 |
| 문서 ID | 선택한 숫자 행을 실제 원문과 연결하는 식별자 | review_id |
| 토픽 | 여러 문서에서 공통으로 발견하는 주제 | 비슷한 문서를 묶은 뒤 대표 키워드·원문으로 해석 |
| RAG | 관련 문서를 검색해 답변 생성에 활용하는 방식 | 검색 결과를 LLM에 전달하는 과정 포함 |

### 글자 수·벡터 차원·벡터 크기는 서로 다릅니다

512글자의 리뷰를 숫자 두 개로 표현한다고 가정할 수 있습니다. 여기서 **512는 원문 길이**, **2는 벡터 차원**, **√(0.8²+0.6²)=1은 벡터 크기**입니다. 벡터가 2차원이라고 글이 두 문장이라는 뜻도, 크기가 1이라고 리뷰가 한 글자라는 뜻도 아닙니다.

2차원 벡터는 원점에서 해당 좌표로 향하는 화살표로 생각하면 방향과 크기를 구분하기 쉽습니다. 다만 이번 첫째·둘째 좌표에 ‘보온성 점수’, ‘사이즈 점수’처럼 의미를 붙이지 않습니다. 실제 임베딩도 각 숫자가 사람이 미리 정한 특정 단어·속성 하나를 그대로 뜻한다고 가정하지 않습니다.

### TF-IDF와 임베딩을 목적에 맞게 구분하기

| 구분 | TF-IDF | 텍스트 임베딩 |
|---|---|---|
| 공통점 | 텍스트를 숫자 벡터로 표현 | 텍스트를 숫자 벡터로 표현 |
| 주요 기준 | 단어 빈도와 문서 집합에서의 희소성 | 모델이 학습한 텍스트 특징 |
| 숫자를 읽는 방식 | 각 열을 표현 이름과 연결해 가중치 확인 | 각 좌표에 사람이 임의로 단어 이름을 붙여 해석하지 않음 |
| 이번 강의에서 연결할 목적 | 문서를 특징짓는 키워드 후보 확인 | 표현이 달라도 관련된 의미의 문서 검색 |

임베딩을 사용해도 부정 표현·조건·사실관계를 원문으로 확인해야 합니다. 유사한 주제를 다루는 것과 같은 사실을 말하는 것은 다릅니다. 어떤 방법이 모든 목적에서 항상 더 좋다고 외우기보다, 필요한 결과가 키워드인지 관련 문서인지부터 구분하세요.

### 수식과 기호를 처음 읽는 방법

질문 벡터를 `q=[q1,q2]`, 문서 벡터를 `d=[d1,d2]`라고 하면 아래 순서로 계산합니다. q1·q2의 숫자 1·2는 성분 위치를 구분하는 표시입니다.

| 단계 | 계산 | 말로 읽기 |
|---|---|---|
| 1. 내적 | `q1×d1 + q2×d2` | 같은 위치의 수끼리 곱하고 합침 |
| 2. 질문 크기 | `√(q1²+q2²)` | 각 성분을 제곱해 더한 뒤 제곱근 계산 |
| 3. 문서 크기 | `√(d1²+d2²)` | 문서에도 같은 크기 계산 적용 |
| 4. 코사인 유사도 | `내적 / (질문 크기 × 문서 크기)` | 내적에서 크기의 영향을 나누어 방향 비교 |

`0.6²`는 `0.6×0.6=0.36`이라는 뜻입니다. `√1=1`에서 √는 제곱하면 해당 수가 되는 음이 아닌 값을 구하는 표시입니다. 벡터 크기를 구할 때는 제곱의 합에서 계산이 끝나지 않고 **제곱근까지** 구합니다. 코드에서는 `np.linalg.norm()`이 이 크기 계산을 수행합니다.

코사인 유사도는 크기가 0이 아닌 벡터 사이에서 정의됩니다. 모든 성분이 0이면 분모가 0이 되어 이 식으로 방향을 비교할 수 없습니다. 이번에 제공한 벡터는 모두 크기가 0이 아니므로 별도 예외 처리 구현은 요구하지 않습니다.

### 파이썬 표현과 계산 대상

| 코드 표현 | 쉬운 뜻 | 이번에 주의할 점 |
|---|---|---|
| `np.array([...])` | 숫자 목록을 NumPy 배열로 만듦 | 글을 임베딩하는 모델 호출이 아님 |
| `@` | 배열 사이의 행렬 곱 | 이번 모양에서는 각 문서 행과 질문의 내적을 계산 |
| `*`, `/` | 곱하기, 나누기 | 이 코드에서는 문서별로 대응하는 크기와 내적을 계산 |
| `np.linalg.norm(..., axis=1)` | 문서 배열의 각 행 안에서 크기를 구함 | 행마다 결과 하나씩 나옴 |
| `np.argsort(점수)` | 작은 점수부터 나열했을 때의 위치 번호 반환 | 점수값이나 리뷰 ID 자체가 아님 |
| `[::-1]` | 순서를 뒤집어 선택 | 작은 순서의 위치를 큰 순서로 바꿀 때 사용 |
| `[:K]` | 앞에서 K개 선택 | 끝 위치 K는 포함하지 않음 |
| `.iloc[위치목록]` | 지정한 위치와 순서대로 행 선택 | 위치 번호로 실제 리뷰 ID를 찾아냄 |
| `.reset_index(drop=True)` | 선택 결과의 행 번호를 0부터 새로 붙임 | review_id는 유지 |
| `.round(2)` | 소수 둘째 자리까지 반올림한 결과 만들기 | 이번에는 출력할 때 사용하며 순위 계산은 원래 값 사용 |

**크기 계산과 후보 선택을 나누어 읽으세요.** 앞부분은 주어진 좌표의 점수를 구하는 수학 계산이고, 뒷부분은 그 점수를 실제 리뷰의 ID·원문과 연결하는 표 처리입니다.

## 실습 준비

다음은 데이터 확인용 **제공 코드**입니다. 그대로 실행하세요. 파일 읽기·조회 함수 작성 자체는 평가하지 않습니다.

1. 세 파일의 크기, 컬럼명, 자료형, 결측치를 확인합니다.
2. 원본 순서의 앞 3개 리뷰를 후보로 사용하고, 본문 결측·빈 본문을 확인합니다.
3. 원문은 유지하고 앞뒤 공백만 정리한 `clean_text`를 만듭니다.
4. 후보 리뷰의 글자 수와 출처·분야 분포를 확인합니다.
5. 같은 ID에 해당하는 전체 라벨과 속성별 주석을 준비합니다.

> 문서 3개는 교안의 소규모 계산 예시에 맞춘 후보입니다. 전체 데이터의 대표 표본이 아니며, 3개 리뷰로 실제 검색 품질이나 전체 토픽 구성을 평가하지 않습니다.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(".")
# 예: DATA_DIR = Path("C:/Users/jihun/OneDrive/Desktop/텍스트분석/DATA")
reviews = pd.read_csv(DATA_DIR / "reviews.csv")
review_labels = pd.read_csv(DATA_DIR / "review_labels.csv")
aspect_labels = pd.read_csv(DATA_DIR / "aspect_labels.csv")

for name, table in [("reviews", reviews), ("review_labels", review_labels),
                    ("aspect_labels", aspect_labels)]:
    print(f"\n[{name}] {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())

candidates = reviews.head(3).copy()
print("\n선택한 본문의 결측 수:", candidates["review_text"].isna().sum())
candidates = candidates.dropna(subset=["review_text"]).copy()
candidates["clean_text"] = candidates["review_text"].str.strip()
print("정제 후 빈 본문 수:", candidates["clean_text"].eq("").sum())
candidates = candidates[candidates["clean_text"] != ""].reset_index(drop=True)
print("후보 리뷰 수:", len(candidates))
print(candidates[["review_id", "source", "domain", "category", "product_name"]].to_string(index=False))
print("\n본문 글자 수:", candidates["clean_text"].str.len().tolist())
print("\n분야 분포:")
print(candidates["domain"].value_counts(dropna=False).to_string())
print("\n출처 분포:")
print(candidates["source"].value_counts(dropna=False).to_string())

candidate_ids = candidates["review_id"].tolist()
candidate_review_labels = review_labels[review_labels["review_id"].isin(candidate_ids)].copy()
candidate_aspect_labels = aspect_labels[aspect_labels["review_id"].isin(candidate_ids)].copy()
print("\n연결된 전체 라벨 행 수:", len(candidate_review_labels))
print("연결된 속성별 주석 행 수:", len(candidate_aspect_labels))

def show_review_context(review_id):
    row = candidates[candidates["review_id"] == review_id].iloc[0]
    print("\n리뷰 ID:", review_id)
    print("상품명:", row["product_name"])
    print("원문:", row["review_text"])
    print("\n기존 전체 라벨:")
    print(candidate_review_labels[candidate_review_labels["review_id"] == review_id].to_string(index=False))
    print("\n기존 속성별 주석:")
    print(candidate_aspect_labels[candidate_aspect_labels["review_id"] == review_id]
          .sort_values("aspect_no").to_string(index=False))



[reviews] 99,992행 × 6열
                자료형  결측치
review_id     int64    0
source          str    0
domain          str    0
category        str    0
product_name    str   16
review_text     str    0

[review_labels] 99,992행 × 2열
                    자료형   결측치
review_id         int64     0
sentiment_code  float64  1143

[aspect_labels] 373,167행 × 5열
                  자료형  결측치
review_id       int64    0
aspect_no       int64    0
aspect            str    0
evidence_text     str    0
sentiment_code  int64    0

선택한 본문의 결측 수: 0
정제 후 빈 본문 수: 0
후보 리뷰 수: 3
 review_id source domain category  product_name
   1024338    SNS     패션     여성의류   OO 경량 다운 자켓
   1024477    SNS     패션     여성의류 OO 아** 구스코트  
   1025044    SNS     패션     여성의류 OO 아** 구스코트  

본문 글자 수: [512, 464, 314]

분야 분포:
domain
패션    3

출처 분포:
source
SNS    3

연결된 전체 라벨 행 수: 3
연결된 속성별 주석 행 수: 22


### 준비 코드와 출력 연결하기

DATA_DIR의 `Path(".")`는 현재 작업 폴더를 뜻합니다. 파일이 다른 곳에 있으면 CSV 세 개가 있는 폴더를 지정합니다. `DATA_DIR / 파일명`은 폴더와 파일명을 연결합니다.

준비 코드는 앞 세 리뷰를 선택하고, 본문 결측과 빈 본문을 확인합니다. clean_text에는 앞뒤 공백을 정리한 문자열을 넣고 review_text는 원문으로 남깁니다. 출력의 글자 수는 벡터 차원이 아니며, 라벨·주석의 행 수는 후보 리뷰 수와 다를 수 있습니다.

후보 표의 첫·둘째·셋째 행과 뒤에서 제시한 문서 벡터의 행을 같은 순서로 연결하세요. 제공 조회 함수는 ID에 맞는 원문·기존 라벨을 출력하는 방법을 정의합니다. 벡터 생성·감정 예측·답변 생성 기능은 없습니다. 준비 코드의 구문별 상세 설명은 정답 및 해설에 있습니다.

---
## 필수 1. 코사인 유사도의 계산 원리

### 문제 1-1. 내적과 벡터 크기로 방향 비교하기

#### 문제 설명

교안 연습문제에 제시된 벡터를 사용합니다. 실제 텍스트 임베딩이 아니라 **계산 원리를 설명하기 위한 가상 벡터**입니다.

| 대상 | 벡터 |
|---|---|
| 질문 Q | [1.0, 0.0] |
| 문서 A | [0.6, 0.8] |
| 문서 B | [0.0, 1.0] |

코사인 유사도는 `두 벡터의 내적 / (질문 벡터의 크기 × 문서 벡터의 크기)`로 계산합니다.

#### 요구사항

1. Q를 `q_vector`, A·B를 행으로 모은 배열을 `example_vectors`로 만드세요.
2. `example_vectors @ q_vector`로 문서별 내적을 계산하세요.
3. `np.linalg.norm(q_vector)`와 `np.linalg.norm(example_vectors, axis=1)`로 질문 및 문서별 크기를 구하세요.
4. 내적을 두 크기의 곱으로 나누어 코사인 유사도를 계산하세요.
5. `문서`, `내적`, `질문_크기`, `문서_크기`, `코사인유사도` 컬럼의 표를 만들고 소수점 둘째 자리까지 출력하세요.

#### 해석 질문

**Q1.** Q와 A의 내적, 각 벡터 크기, 코사인 유사도를 수식을 대입하여 설명하세요. A와 B 중 어느 쪽이 Q와 방향이 더 유사한가요?

**Q2.** 코사인 유사도 1·0·-1은 벡터 방향에서 각각 무엇을 의미하나요? 0이면 두 문장의 의미가 반드시 무관하다고 해석해도 되나요?

**Q3.** TF-IDF와 텍스트 임베딩의 공통점과 표현 기준의 차이는 무엇인가요? 키워드 추출과 의미가 비슷한 문의 검색에는 각각 무엇을 우선 고려하나요?

**Q4.** 이번 벡터와 점수가 실제 문장의 의미를 분석한 결과인가요? 실제 검색에서 질문과 문서의 벡터를 만들 때 무엇이 호환되어야 하나요?

#### 제출 결과

- 요구사항 1~5의 코드와 계산 표
- Q1~Q4 답변


### 필수 1 풀이를 시작하기 전에

문서별로 ‘첫째 성분끼리 곱하기 → 둘째 성분끼리 곱하기 → 합하기’를 적어보세요. 그다음 질문과 문서의 크기를 각각 구해 분모를 만듭니다. 제곱의 합과 제곱근을 구분하세요.

문서 배열에서는 한 행이 한 문서입니다. `axis=1`로 행마다 크기를 구하고 내적과 같은 문서끼리 나누어야 합니다. 최종 점수뿐 아니라 중간값의 뜻도 함께 설명하세요. 수치의 정답과 자세한 대입 과정은 정답 및 해설에 있습니다.

In [2]:
# 필수 1 코드를 작성하세요.

# 1. 질문 벡터와 문서 벡터 (교안의 가상 벡터, 실제 임베딩 아님)
q_vector = np.array([1.0, 0.0])
example_vectors = np.array([[0.6, 0.8],   # 문서 A
                            [0.0, 1.0]])  # 문서 B

# 2. 문서별 내적
dot_products = example_vectors @ q_vector

# 3. 질문 크기, 문서별 크기 (행마다 계산하려고 axis=1)
q_norm = np.linalg.norm(q_vector)
doc_norms = np.linalg.norm(example_vectors, axis=1)

# 4. 코사인 유사도 = 내적 / (질문 크기 x 문서 크기)
cosine_scores = dot_products / (q_norm * doc_norms)

# 5. 결과 표 (소수점 둘째 자리까지 표시)
example_result = pd.DataFrame({
    "문서": ["A", "B"],
    "내적": dot_products,
    "질문_크기": q_norm,
    "문서_크기": doc_norms,
    "코사인유사도": cosine_scores,
})
print(example_result.round(2).to_string(index=False))

문서  내적  질문_크기  문서_크기  코사인유사도
 A 0.6    1.0    1.0     0.6
 B 0.0    1.0    1.0     0.0


### 필수 1 답변 작성란

**Q1.**

- 답변: Q와 A의 내적은 1.0 x 0.6 + 0.0 x 0.8 = 0.6이다. Q의 크기는 제곱근(1.0의 제곱 + 0.0의 제곱) = 제곱근(1) = 1이고, A의 크기는 제곱근(0.6의 제곱 + 0.8의 제곱) = 제곱근(0.36 + 0.64) = 제곱근(1) = 1이다. 따라서 코사인 유사도는 0.6 / (1 x 1) = 0.6이다. B는 내적이 1.0 x 0.0 + 0.0 x 1.0 = 0이고 크기는 1이라 코사인 유사도가 0이다. 그래서 A가 Q와 방향이 더 유사하다.

**Q2.**

- 답변: 1은 두 벡터가 완전히 같은 방향, 0은 서로 수직(직각)이라 방향상 겹치는 부분이 없음, -1은 정반대 방향을 뜻한다. 하지만 0이라고 해서 두 문장의 의미가 반드시 무관하다고 해석하면 안 된다. 이번처럼 사람이 정한 가상 벡터라면 실제 의미와 관계가 없고, 실제 임베딩이라도 모델이나 표현 방식에 따라 점수가 달라질 수 있어서 원문을 확인해야 한다.

**Q3.**

- 답변: 공통점은 둘 다 텍스트를 숫자 벡터로 표현해서 비교할 수 있게 만든다는 것이다. 차이는 TF-IDF는 단어 빈도와 문서 집합에서의 희소성을 기준으로 하고 각 열이 특정 표현에 대응하지만, 임베딩은 모델이 학습한 텍스트 특징을 기준으로 하고 각 좌표에 단어 이름을 붙여 해석하지 않는다는 점이다. 키워드 추출에는 TF-IDF를, 표현은 달라도 의미가 비슷한 문의 검색에는 텍스트 임베딩을 우선 고려하겠다.

**Q4.**

- 답변: 아니다. 이번 벡터는 사람이 정한 2차원 가상 값이고, 점수도 그 벡터에 대한 수학적 계산 결과일 뿐이라 실제 문장의 의미를 분석한 결과가 아니다. 실제 검색에서는 질문과 문서를 같은 임베딩 모델로 변환해야 하고, 그래서 벡터 차원과 벡터 공간이 같아야 비교할 수 있다. 질문과 문서의 언어도 그 모델이 지원하는지 확인해야 한다.


---
## 필수 2. Top-K 순위와 실제 리뷰 원문 연결하기

### 문제 2-1. 계산한 순위를 실제 문서에 연결해 확인하기

#### 문제 설명

후보 리뷰 3개를 대상으로 순위 계산과 원문 확인을 연습합니다. 교안의 예시와 같은 문서 벡터를 아래처럼 후보에 대응시켰다고 **가정**합니다. 임베딩 모델을 실행하지 않았으므로 실제 의미 검색 결과가 아닙니다.

질문 문장: `가볍게 입기 좋은 패딩 후기를 찾고 있어요.`

| 대상 | 후보 위치 | 가상 벡터 |
|---|---:|---|
| 질문 | 해당 없음 | [1.0, 0.0] |
| 첫 번째 리뷰 | 0 | [0.8, 0.6] |
| 두 번째 리뷰 | 1 | [0.6, 0.8] |
| 세 번째 리뷰 | 2 | [0.0, 1.0] |

벡터는 실제 CSV를 변환한 값이 아니라 제공된 계산 재료입니다. **주어진 벡터에서의 순위가 맞는지**와 **읽은 리뷰가 질문에 도움이 되는지**를 구분하여 답하세요.

#### 요구사항

1. 질문을 `query`, 질문 벡터를 `query_vector`, 문서 벡터를 `document_vectors`에 저장하세요. 문서 벡터의 행 순서를 `candidates`의 리뷰 순서와 같게 유지하세요.
2. 교안의 NumPy 코드처럼 내적, 질문 크기, 문서별 크기를 구하고 `similarities`를 계산하세요.
3. 후보 ID·상품명과 `가상벡터_코사인유사도`를 함께 담은 `all_scores` 표를 만들어 출력하세요.
4. `top_k = 2`로 설정하고 `np.argsort(similarities)[::-1][:top_k]`로 `top_indices`를 구하세요. 결과는 리뷰 ID가 아니라 **후보의 위치 번호**입니다.
5. `all_scores.iloc[top_indices]`로 Top-2 결과 표 `top2_results`를 만들고 순위, 리뷰 ID, 상품명, 점수를 출력하세요.
6. Top-2 리뷰 각각에 대해 `show_review_context()`를 실행하여 원문·기존 전체 라벨·속성별 근거를 확인하세요.

#### 해석 질문

**Q1.** 주어진 가상 벡터에서 Top-2의 리뷰 ID와 점수는 무엇인가요? 점수 0.80을 “정답일 확률 80%”라고 읽어도 되나요?

**Q2.** 1위 리뷰에서 가볍게 입는 용도와 관련된 근거 하나, 착용 조건에 따른 제한 하나를 적으세요. 2위 리뷰에서도 질문에 참고할 수 있는 근거 하나를 적으세요.

**Q3.** Top-K에 포함되면 반드시 적절한 답변 근거인가요? 0점인 세 번째 리뷰를 실제 의미상 무관하다고 결론 내릴 수 있나요?

**Q4.** 실제 임베딩 검색에서 원문을 추가로 확인해야 하는 이유를 `환불을 받았습니다`와 `환불을 받지 못했습니다`의 차이로 설명하세요. 기존 sentiment_code는 이번 검색 점수와 어떤 관계인가요?

#### 제출 결과

- 전체 후보 점수 표와 Top-2 표
- Top-2의 원문·라벨·근거 출력
- Q1~Q4 답변


### 필수 2 풀이 도움말

후보 표의 순서와 문서 벡터의 행 순서를 먼저 맞춥니다. `np.argsort()`가 반환한 것은 점수가 아니라 위치 번호이므로, 뒤집기와 앞 K개 선택을 각각 확인하세요.

결과 표에는 순위·실제 리뷰 ID·점수를 함께 남깁니다. 선택된 원문에서는 질문과 연결되는 표현뿐 아니라 제한 조건도 찾습니다. 점수가 높다는 계산 결과와 질문에 도움이 된다는 해석을 나누어 적으세요.

In [3]:
# 필수 2 코드를 작성하세요.

# 1. 질문 문장, 가상 질문 벡터, 가상 문서 벡터 (행 순서 = candidates 순서)
query = "가볍게 입기 좋은 패딩 후기를 찾고 있어요."
query_vector = np.array([1.0, 0.0])
document_vectors = np.array([
    [0.8, 0.6],  # 위치 0: 첫 번째 리뷰
    [0.6, 0.8],  # 위치 1: 두 번째 리뷰
    [0.0, 1.0],  # 위치 2: 세 번째 리뷰
])
print("질문:", query)
print("후보 순서 확인:", candidates["review_id"].tolist())

# 2. 내적, 질문 크기, 문서별 크기로 코사인 유사도 계산
dot_products = document_vectors @ query_vector
query_norm = np.linalg.norm(query_vector)
doc_norms = np.linalg.norm(document_vectors, axis=1)
similarities = dot_products / (query_norm * doc_norms)

# 3. 후보 ID, 상품명, 점수를 함께 담은 표
all_scores = candidates[["review_id", "product_name"]].copy()
all_scores["가상벡터_코사인유사도"] = similarities
print("\n[전체 후보 점수]")
print(all_scores.round(2).to_string(index=False))

# 4. Top-K 위치 번호 (리뷰 ID가 아니라 후보 안에서의 위치)
top_k = 2
top_indices = np.argsort(similarities)[::-1][:top_k]
print("\ntop_indices (후보 위치 번호):", top_indices)

# 5. 위치 번호로 실제 리뷰 행 선택 -> 순위 붙이기
top2_results = all_scores.iloc[top_indices].reset_index(drop=True)
top2_results.insert(0, "순위", range(1, top_k + 1))
print("\n[Top-2 결과]")
print(top2_results.round(2).to_string(index=False))

# 6. Top-2 각각의 원문, 기존 전체 라벨, 속성별 근거 확인
for review_id in top2_results["review_id"]:
    show_review_context(review_id)

질문: 가볍게 입기 좋은 패딩 후기를 찾고 있어요.
후보 순서 확인: [1024338, 1024477, 1025044]

[전체 후보 점수]
 review_id  product_name  가상벡터_코사인유사도
   1024338   OO 경량 다운 자켓          0.8
   1024477 OO 아** 구스코트            0.6
   1025044 OO 아** 구스코트            0.0

top_indices (후보 위치 번호): [0 1]

[Top-2 결과]
 순위  review_id  product_name  가상벡터_코사인유사도
  1    1024338   OO 경량 다운 자켓          0.8
  2    1024477 OO 아** 구스코트            0.6

리뷰 ID: 1024338
상품명: OO 경량 다운 자켓
원문: 안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요. 색상도 디자인도 무난해서 사무실에 두고 입기도 좋고, 평소에 가볍게 나갈때 코디하기도 좋아요. 앞으로 코트를 입거나 할때 안에 내피로 활용하기도 딱일 듯요! 맘같아선 색깔별로 쟁이고 싶습니다.  20대와 50대 모두 아우르는 디자인이 무난하고 깔끔하면서 고급스러운 옷 추천합니다~~~  설명 끝! 좀 도움이 되셨나요? 그러면 좋아요 꾹 눌러 주시고 전 이만 총총총~~ 항상 여러 이웃님들께 감사드립니다. 

기존 전체 라벨:
 review_id  sentiment_code
   1024338 

### 필수 2 답변 작성란

**Q1.**

- 답변: Top-1은 1024338(점수 0.80), Top-2는 1024477(점수 0.60)이다. 0.80은 가상 벡터 두 개의 방향이 얼마나 비슷한지를 나타낸 코사인 유사도이지 정답일 확률 80%라는 뜻이 아니다. 이번에는 확률을 계산하지 않았다.

**Q2.**

- 답변: 1위 1024338에서 가볍게 입는 용도와 관련된 근거는 '평소에 가볍게 나갈때 코디하기도 좋아요.'(활용성, 코드 1)이고, 상품명도 '경량 다운 자켓'이다. 착용 조건에 따른 제한은 '영하로 내려가는 날씨에는 이것만 입기엔 얇지만'(두께, 코드 -1)으로, 한겨울에 이것만 입기엔 얇다는 뜻이다. 2위 1024477에서는 '간편하게 니트 티셔츠에 입기에는 딱입니다.'(활용성, 코드 1)를 참고할 수 있다. 다만 같은 리뷰에 '안에 두껍게 입기엔 좀 불편하더라구요.'라는 제한도 있고, 상품이 경량 패딩이 아닌 구스코트라는 점도 같이 봐야 한다.

**Q3.**

- 답변: 아니다. Top-K는 주어진 점수로 후보를 고른 것일 뿐이라서, 원문에 질문 조건(가볍게 입기 좋음)을 뒷받침하는 내용이 있는지 따로 확인해야 답변 근거로 쓸 수 있다. 세 번째 리뷰 1025044의 0점은 가상 벡터 [0.0, 1.0]이 질문 벡터 [1.0, 0.0]과 수직이라 나온 계산 결과이다. 실제 원문은 '롱패딩 찾고 계신 분을 위한 후기'라서 패딩이라는 주제는 관련이 있다. 원문에서 가볍다는 언급은 찾을 수 없어서 이 질문의 근거로는 부족하지만, 그렇다고 '의미상 무관하다'고 결론 내릴 수는 없다.

**Q4.**

- 답변: '환불을 받았습니다'와 '환불을 받지 못했습니다'는 '환불', '받다'라는 단어와 주제가 거의 같아서 실제 임베딩 벡터도 비슷하게 나와 유사도가 높게 나올 수 있다. 하지만 '못했습니다'라는 부정 표현 때문에 사실은 정반대이다. 그래서 검색 점수가 높아도 원문을 읽고 사실관계와 조건을 확인해야 한다. 기존 sentiment_code는 CSV에 원래 있던 리뷰 전체 라벨이고 이번 검색 점수와는 관계가 없다. 실제로 후보 3개의 코드는 모두 1.0인데 점수는 0.80, 0.60, 0.00으로 달랐다.


---
## 과제 1. 다른 질문의 Top-K 계산과 활용 방법 구분

### 문제 3-1. 검색 후보를 해석하고 BERTopic·RAG의 역할 설명하기

#### 문제 설명

필수 2의 후보 리뷰 3개와 `document_vectors`를 그대로 사용합니다. 이번에는 다음 질문에 가상 질문 벡터 `[0.0, 1.0]`을 대응시켰다고 가정하고 같은 절차를 수행하세요.

질문 문장: `사이즈가 여유 없이 맞는 패딩 후기를 찾고 있어요.`

이 벡터 역시 모델이 질문에서 생성한 값이 아닙니다. 질문 문장을 바꾸면 벡터가 자동으로 바뀌는 코드를 구현하는 것이 아닙니다. **별도로 주어진 질문 벡터로 계산을 반복하는 과제**입니다.

#### 요구사항

1. 질문을 `task_query`, 가상 질문 벡터를 `task_query_vector`에 저장하세요.
2. 기존 문서 벡터와의 내적·벡터 크기를 계산하고 `task_similarities`를 구하세요.
3. 후보 ID·상품명·점수 표 `task_all_scores`를 출력하고, 점수가 높은 Top-2의 순위·ID·상품명·점수를 `task_top2`로 출력하세요.
4. 새 1위 리뷰의 원문, 기존 전체 라벨, 속성별 근거를 `show_review_context()`로 출력하세요.
5. 해당 리뷰에서 사이즈와 관련된 실제 근거를 확인하고, 아래 질문에 답하세요. BERTopic·RAG는 개념만 설명하며 실행하지 않습니다.

#### 해석 질문

**Q1.** 새 Top-2의 ID·점수는 무엇인가요? 필수 2와 순위가 달라진 계산상의 이유는 무엇이며, 새 1위 리뷰에서 사이즈에 관해 확인할 수 있는 실제 근거는 무엇인가요?

**Q2.** “이 질문과 비슷한 리뷰 찾기”와 “전체 리뷰에서 주요 주제 발견하기”에 각각 적합한 작업과 결과 형태를 설명하세요. 이 과제에서 실제 임베딩 모델이나 토픽 모델을 실행했나요?

**Q3.** 교안의 BERTopic 처리 단계 5개를 순서대로 쓰고 역할을 설명하세요. c-TF-IDF는 무엇을 위해 사용하나요? 생성된 토픽을 기존 aspect·sentiment_code와 동일하다고 간주해도 되나요?

**Q4.** RAG의 기본 흐름과 임베딩·코사인 유사도·문서 검색·LLM의 역할을 설명하세요. 이 과제에서 Top-2를 출력한 것으로 RAG 답변 생성까지 완료했다고 말할 수 있나요?

#### 제출 결과

- 새 질문의 전체 점수 표와 Top-2 표
- 새 1위 리뷰의 원문·라벨·근거 출력
- Q1~Q4 답변

새 파일이나 외부 서비스 결과를 제출할 필요는 없습니다. 코드 결과와 답변을 노트북에 남기세요.


### 과제 시작 전 확인

필수 2의 문서 벡터와 후보 순서를 그대로 유지하고, 문제에 따로 주어진 새 질문 벡터를 사용합니다. 질문 문자열만 바꾸는 것으로는 계산 입력이 바뀌지 않습니다.

새 순위를 구한 뒤 결과 표의 첫 리뷰 ID로 원문을 확인합니다. 원래 후보의 첫 행과 새 1위를 혼동하지 마세요. BERTopic·RAG 문항은 목적과 단계의 역할을 설명하는 문제이며, 모델 설치나 외부 서비스 실행은 필요하지 않습니다.

In [4]:
# 과제 1 코드를 작성하세요.

# 1. 새 질문과 별도로 주어진 가상 질문 벡터
task_query = "사이즈가 여유 없이 맞는 패딩 후기를 찾고 있어요."
task_query_vector = np.array([0.0, 1.0])
print("질문:", task_query)

# 2. 기존 document_vectors 그대로 사용해서 내적, 크기, 코사인 유사도 계산
task_dot_products = document_vectors @ task_query_vector
task_query_norm = np.linalg.norm(task_query_vector)
task_doc_norms = np.linalg.norm(document_vectors, axis=1)
task_similarities = task_dot_products / (task_query_norm * task_doc_norms)

# 3. 전체 점수 표와 Top-2 표
task_all_scores = candidates[["review_id", "product_name"]].copy()
task_all_scores["가상벡터_코사인유사도"] = task_similarities
print("\n[새 질문의 전체 후보 점수]")
print(task_all_scores.round(2).to_string(index=False))

task_top_indices = np.argsort(task_similarities)[::-1][:top_k]
print("\ntask_top_indices (후보 위치 번호):", task_top_indices)
task_top2 = task_all_scores.iloc[task_top_indices].reset_index(drop=True)
task_top2.insert(0, "순위", range(1, top_k + 1))
print("\n[새 질문의 Top-2 결과]")
print(task_top2.round(2).to_string(index=False))

# 4. 새 1위 리뷰 (결과 표의 첫 행 ID로 조회)
task_first_id = task_top2["review_id"].iloc[0]
show_review_context(task_first_id)

# 5. 새 1위 리뷰에서 사이즈 관련 속성별 근거만 따로 확인
size_rows = candidate_aspect_labels[
    (candidate_aspect_labels["review_id"] == task_first_id)
    & (candidate_aspect_labels["aspect"] == "사이즈")
]
print("\n[새 1위 리뷰의 사이즈 관련 근거]")
print(size_rows[["aspect_no", "aspect", "evidence_text", "sentiment_code"]].to_string(index=False))

질문: 사이즈가 여유 없이 맞는 패딩 후기를 찾고 있어요.

[새 질문의 전체 후보 점수]
 review_id  product_name  가상벡터_코사인유사도
   1024338   OO 경량 다운 자켓          0.6
   1024477 OO 아** 구스코트            0.8
   1025044 OO 아** 구스코트            1.0

task_top_indices (후보 위치 번호): [2 1]

[새 질문의 Top-2 결과]
 순위  review_id  product_name  가상벡터_코사인유사도
  1    1025044 OO 아** 구스코트            1.0
  2    1024477 OO 아** 구스코트            0.8

리뷰 ID: 1025044
상품명: OO 아** 구스코트  
원문: 오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다.  키 158센티로에 48키로 마른 체형인데 90사이즈가 여유가 별로 없이 딱 맞습니다. 저는 딱 맞게 입기를 좋아해서 아주 만족해요. 와인색이 정말 특이하게 다운된 색인데 정말 고급스러워서 더욱 마음에 듭니다. 가격도 아주 저렴한 편이라 부담없이 사서 입게 되었답니다. 핏되게 나온 옷이라 넉넉하게 입기를 좋아하시는 분은 한 사이즈 올려 입으셔도 좋을 것 같아요. 저는 이 패딩으로 올겨울 아주 따뜻하게 잘 지내게 될 것 같습니다.   새로운 패딩 장만하고 올 겨울 따스하게 보내시길요...

기존 전체 라벨:
 review_id  sentiment_code
   1025044             1.0

기존 속성별 주석:
 review_id  aspect_no aspect                                        evidence_text  sentiment_code
   1025044          1    사이즈     키 158센티로에 48키로 마른 체형인데 90사이즈가 여유가 별로 없이 딱 맞습니다.        

### 과제 1 답변 작성란

**Q1.**

- 답변: 새 Top-2는 1위 1025044(1.00), 2위 1024477(0.80)이다. 문서 벡터는 그대로이고 질문 벡터만 [1.0, 0.0]에서 [0.0, 1.0]으로 바뀌어서, 이번에는 둘째 성분이 큰 문서일수록 내적이 커졌다. 1025044의 벡터 [0.0, 1.0]은 새 질문 벡터와 같은 방향이라 1.00이 나왔다. 새 1위 리뷰에서 사이즈에 관한 실제 근거는 '키 158센티로에 48키로 마른 체형인데 90사이즈가 여유가 별로 없이 딱 맞습니다.'(사이즈, 코드 1)와 '핏되게 나온 옷이라 넉넉하게 입기를 좋아하시는 분은 한 사이즈 올려 입으셔도 좋을 것 같아요.'(사이즈, 코드 0)이다. 순위는 가상 벡터의 계산 결과이고, 질문에 맞는 근거가 있다는 것은 원문을 읽고 따로 확인한 것이다.

**Q2.**

- 답변: '이 질문과 비슷한 리뷰 찾기'는 유사 문서 검색이 적합하다. 질문과 리뷰를 임베딩한 뒤 코사인 유사도로 비교하고, 결과는 점수가 높은 Top-K 리뷰 목록(리뷰 ID, 점수, 원문)이다. '전체 리뷰에서 주요 주제 발견하기'는 BERTopic 같은 토픽 모델링이 적합하고, 결과는 토픽 번호, 토픽별 대표 키워드, 각 리뷰가 속한 토픽이다. 이번 과제에서는 실제 임베딩 모델이나 토픽 모델을 실행하지 않았고 가상 벡터로 코사인 유사도만 계산했다.

**Q3.**

- 답변: ① 문서 임베딩: 각 문서를 의미를 담은 벡터로 변환한다. ② 차원 축소(UMAP): 고차원 벡터를 군집을 찾기 쉬운 낮은 차원으로 줄인다. ③ 클러스터링(HDBSCAN): 가까운 문서끼리 묶어서 토픽 후보 그룹을 만든다. ④ 토큰화(CountVectorizer): 각 그룹 문서의 단어를 세어 단어 빈도 표를 만든다. ⑤ c-TF-IDF: 그룹별 대표 단어를 뽑아 토픽을 표현한다. c-TF-IDF는 같은 클러스터의 문서를 하나의 묶음으로 보고 TF-IDF를 계산해서, 그 토픽을 다른 토픽과 구별해 주는 대표 키워드를 찾기 위해 사용한다. 생성된 토픽을 기존 aspect나 sentiment_code와 같다고 보면 안 된다. 토픽은 모델이 비슷한 문서를 묶고 키워드를 붙인 결과이고, aspect는 사람이 근거 문장에 붙인 속성 라벨, sentiment_code는 감정 코드라서 만든 기준과 단위가 다르다. 토픽의 의미는 대표 키워드와 원문을 읽고 사람이 해석해야 한다.

**Q4.**

- 답변: RAG의 기본 흐름은 질문 입력 → 질문 임베딩 → 미리 임베딩해 둔 문서들과 코사인 유사도 비교 → 관련 문서 Top-K 검색 → 검색한 문서를 질문과 함께 LLM에 전달 → LLM이 그 문서를 근거로 답변 생성이다. 임베딩은 질문과 문서를 비교 가능한 벡터로 바꾸고, 코사인 유사도는 두 벡터의 방향을 비교해 관련 정도를 점수로 나타내고, 문서 검색은 점수가 높은 근거 후보를 고르고, LLM은 검색된 문서를 읽고 답변을 작성한다. 이번 과제는 가상 벡터로 Top-2를 뽑고 원문을 확인한 것까지라서, LLM에 전달하거나 답변을 생성하지 않았으므로 RAG 답변 생성까지 완료했다고 말할 수 없다.


## 개념 정리 — 이번 강의에서 구분할 것

| 구분 | 핵심 기준 |
|---|---|
| 원문 길이 / 벡터 차원 / 벡터 크기 | 글자 수 / 숫자 개수 / 좌표로 계산한 길이 |
| 내적 / 코사인 유사도 | 성분별 곱의 합 / 내적을 두 크기의 곱으로 나눈 값 |
| 벡터 점수 / 정답 확률 | 방향 유사도를 나타낸 수치 / 이번에는 계산하지 않은 확률 |
| 순위 / 후보 위치 / 리뷰 ID | 결과에서 몇 번째인가 / 원래 배열의 어느 위치인가 / 어떤 실제 리뷰인가 |
| 질문 문자열 / 질문 벡터 | 사람이 읽는 문장 / 계산식에 직접 넣는 좌표 |
| 가상 벡터 / 실제 임베딩 | 사람이 지정한 수 / 모델이 텍스트로부터 생성한 표현 |
| 검색 후보 / 답변 근거 | 점수로 선택한 문서 / 원문·조건을 확인하여 사용할 수 있다고 판단한 내용 |
| 기존 라벨 / 새 유사도 | CSV에서 가져온 코드 / 이번 좌표로 계산한 점수 |
| 검색 / 토픽 발견 / 답변 생성 | 후보 선택 / 전체 문서의 주제 파악 / 검색 자료를 활용한 응답 작성 |

결과를 보고할 때는 **어떤 벡터로 계산했는지 → 어떤 ID가 선택됐는지 → 원문에서 무엇을 확인했는지**를 연결하세요. 가상 점수가 높다는 사실을 실제 모델의 성능이나 제품 추천의 확실성으로 바꾸어 설명하지 않습니다.

---
## 실습 마무리

1. 단어 표현이 다른 문서의 의미를 비교할 때 임베딩을 고려하는 이유는 무엇인가요?
2. 코사인 유사도를 구하려면 어떤 중간값이 필요한가요?
3. Top-K 후보를 선택한 뒤 무엇을 근거로 실제 관련성을 확인했나요?
4. 이번 실습에서 실제 데이터인 부분과 가정한 부분은 무엇인가요?
5. 유사 문서 검색, BERTopic, RAG의 목적은 어떻게 다른가요?

각 질문에 1~3문장으로 답하세요.


### 실습 마무리 답변 작성란

1. '배송이 빨라서 만족'과 '금방 도착해서 좋았다'처럼 단어가 달라도 의미가 비슷할 수 있는데, TF-IDF는 같은 표현만 같은 열로 봐서 이를 잡기 어렵다. 임베딩은 모델이 학습한 특징으로 의미를 벡터로 표현해서 표현이 달라도 비교할 수 있다.

2. 질문과 문서의 내적, 질문 벡터의 크기, 문서 벡터의 크기가 필요하다. 크기는 각 성분을 제곱해서 더한 뒤 제곱근을 구하고, 내적을 두 크기의 곱으로 나누면 코사인 유사도가 된다.

3. Top-K의 위치 번호로 review_id를 찾고 show_review_context()로 원문과 속성별 근거 문장을 읽어서, 질문 조건(가볍게 입기 좋음, 사이즈가 여유 없이 맞음)을 직접 뒷받침하는 표현이 있는지 확인했다.

4. 리뷰 본문, review_id, 상품명, 기존 라벨, 근거 문장은 CSV의 실제 데이터이다. 질문 벡터와 문서 벡터, 그리고 그것으로 계산한 점수와 순위는 계산 연습을 위해 가정한 값이다.

5. 유사 문서 검색은 질문과 비슷한 문서 몇 개를 찾는 것, BERTopic은 전체 문서에서 주요 주제를 발견하는 것, RAG는 검색한 문서를 LLM에 넘겨 근거 있는 답변을 생성하는 것이 목적이다.


1. 
-> 같은 의미라도 단어가 다르면 빈도 중심 표현에서 충분히 가깝게 나타나지 않을 수있음
-> 임베딩은 모델이 학습한 특징으로 관련성을 표현하는 데 도움을 줄 수 있다.

2. 
-> 두 벡터의 내적과 각 벡터의 크기가 필요하다.
-> 내적을 두 크기의 곱으로 나누어 방향이 얼마나 비슷한지 계산한다.

3. 
-> 원문과 속성별 근거를 읽어 질문의 목적과 연결되는 표현을 찾았다.
-> '영하의 날씨', '안에 두꺼운 옷', 작성자의 체형, 사이즈처럼 중요한 조건들도 포함되어 있음을 확인 가능
-> 단, 점수만으로 의미나 사실 관계를 확정 짓지 않는다.

4. 
-> CSV의 리뷰 논문, ID, 메타데이터, 기존 라벨은 실제 제공 데이터,
-> 벡터는 계산 연습용 가상 값이고 순위 자체를 벡터 연습용 가상값으로 구한 결과를 의미
-> 실제 임베딩 모델의 성능이나 실제 의미 검색 결과를 측정한 것은 아니다.

5. 
-> 유사 문서 검색은 특정 질문과 관련된 후보를 찾고, BERTopic은 여러 문서에서 그룹과 주요 문제를 발견한다.
-> RAG는 관련 문서를 검색해 LLM의 답변 생성에 활용한다.
-> 모두 임베딩을 활용할 수는 있으나 같은 작업은 아니다.

각 질문에 1~3문장으로 답하세요.